In [ ]:
import numpy as np 
import pandas as pd 

# 1. TẢI DỮ LIỆU
data = pd.read_csv('accepted_2007_to_2018Q4.csv', low_memory=False)
print(f"Dữ liệu gốc: {data.shape[0]:,} dòng × {data.shape[1]} cột")
data.head()

Dữ liệu gốc: 2,260,701 dòng × 151 cột


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
data.describe()

,member_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,dti,delinq_2yrs,fico_range_low,...,deferral_term,hardship_amount,hardship_length,hardship_dpd,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,settlement_amount,settlement_percentage,settlement_term
count,0.0,2.260668e+06,2.260668e+06,2.260668e+06,2.260668e+06,2.260668e+06,2.260664e+06,2.258957e+06,2.260639e+06,2.260668e+06,...,10917.0,10917.000000,10917.0,10917.000000,8651.000000,10917.000000,10917.000000,34246.000000,34246.000000,34246.000000
mean,NaN,1.504693e+04,1.504166e+04,1.502344e+04,1.309283e+01,4.458068e+02,7.799243e+04,1.882420e+01,3.068792e-01,6.985882e+02,...,3.0,155.045981,3.0,13.743886,454.798089,11636.883942,193.994321,5010.664267,47.780365,13.191322
std,NaN,9.190245e+03,9.188413e+03,9.192332e+03,4.832138e+00,2.671735e+02,1.126962e+05,1.418333e+01,8.672303e-01,3.301038e+01,...,0.0,129.040594,0.0,9.671178,375.385500,7625.988281,198.629496,3693.122590,7.311822,8.159980
min,NaN,5.000000e+02,5.000000e+02,0.000000e+00,5.310000e+00,4.930000e+00,0.000000e+00,-1.000000e+00,0.000000e+00,6.100000e+02,...,3.0,0.640000,3.0,0.000000,1.920000,55.730000,0.010000,44.210000,0.200000,0.000000
25%,NaN,8.000000e+03,8.000000e+03,8.000000e+03,9.490000e+00,2.516500e+02,4.600000e+04,1.189000e+01,0.000000e+00,6.750000e+02,...,3.0,59.440000,3.0,5.000000,175.230000,5627.000000,44.440000,2208.000000,45.000000,6.000000
50%,NaN,1.290000e+04,1.287500e+04,1.280000e+04,1.262000e+01,3.779900e+02,6.500000e+04,1.784000e+01,0.000000e+00,6.900000e+02,...,3.0,119.140000,3.0,15.000000,352.770000,10028.390000,133.160000,4146.110000,45.000000,14.000000
75%,NaN,2.000000e+04,2.000000e+04,2.000000e+04,1.599000e+01,5.933200e+02,9.300000e+04,2.449000e+01,0.000000e+00,7.150000e+02,...,3.0,213.260000,3.0,22.000000,620.175000,16151.890000,284.190000,6850.172500,50.000000,18.000000
max,NaN,4.000000e+04,4.000000e+04,4.000000e+04,3.099000e+01,1.719830e+03,1.100000e+08,9.990000e+02,5.800000e+01,8.450000e+02,...,3.0,943.940000,3.0,37.000000,2680.890000,40306.410000,1407.860000,33601.000000,521.350000,181.000000


In [3]:
import re

cols_before = set(data.columns)

# 2. LỰA CHỌN ĐẶC TRƯNG (LOẠI BỎ CÁC CỘT KHÔNG CẦN THIẾT)
# Bước 1: Loại bỏ các cột định danh và text tự do không có giá trị phân tích
drop_dt = ['member_id', 'url', 'desc', 'emp_title', 'zip_code', 'title']
data = data.drop(drop_dt, axis=1, errors='ignore')

# Bước 2: Loại bỏ cột liên quan đến đồng vay (joint), settlement, hardship, investor
a, b, c = [], [], []
for x in data.columns:
    if x.startswith('sec_app_') or x.endswith('_joint'):
        a.append(x)
    if x.startswith('settlement_') or x.startswith('hardship_'):
        b.append(x)
    if x.endswith('_inv'):
        c.append(x)

data.drop(a + b + c, axis=1, inplace=True, errors='ignore')

# Bước 3: Loại bỏ cột hành chính dư thừa (không nằm trong final_33)
admin_drops = [
    'sub_grade', 'pymnt_plan', 'policy_code', 'initial_list_status', 'disbursement_method',
    'deferral_term', 'orig_projected_additional_accrued_interest', 'mo_sin_old_il_acct',
    'num_tl_120dpd_2m', 'last_fico_range_high', 'last_fico_range_low',
    'next_pymnt_d', 'last_credit_pull_d', 'total_rec_late_fee', 'collection_recovery_fee',
    'payment_plan_start_date', 'debt_settlement_flag', 'debt_settlement_flag_date',
    'application_type', 'acc_now_delinq', 'delinq_amnt', 'funded_amnt'
]
data.drop(admin_drops, axis=1, inplace=True, errors='ignore')

# Bước 4: Loại bỏ cột theo prefix pattern (không ảnh hưởng đến final_33)
# Giữ lại: total_acc, total_pymnt, revol_bal, revol_util (có trong final_33)
KEEP_COLS = {
    'total_acc', 'total_pymnt', 'revol_bal', 'revol_util',
    'total_bc_limit', 'id', 'loan_amnt', 'term', 'int_rate', 'grade',
    'emp_length', 'home_ownership', 'annual_inc', 'issue_d', 'loan_status',
    'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line',
    'fico_range_low', 'fico_range_high', 'pub_rec', 'out_prncp',
    'recoveries', 'last_pymnt_d', 'mort_acc', 'il_util', 'max_bal_bc', 'all_util'
}

drop_patterns = [
    'open_', 'num_', 'mo_', 'mths_', 'inq_', 'bc_', 'pct_', 'percent_',
    'total_rec', 'total_bal', 'total_rev', 'total_cu', 'total_il', 'tot_'
]
specific_drops = [
    'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d',
    'last_fico_range_high', 'last_fico_range_low',
    'collections_12_mths_ex_med', 'application_type',
    'chargeoff_within_12_mths', 'delinq_amnt',
    'pub_rec_bankruptcies', 'tax_liens',
    'payment_plan_start_date', 'debt_settlement_flag',
    'debt_settlement_flag_date', 'deferral_term',
    'orig_projected_additional_accrued_interest', 'disbursement_method',
    'acc_now_delinq', 'acc_open_past_24mths', 'avg_cur_bal',
    'funded_amnt', 'installment', 'verification_status'
]

# Chỉ drop nếu KHÔNG nằm trong KEEP_COLS
cols_to_drop = [
    col for col in data.columns
    if col not in KEEP_COLS
    and (any(col.startswith(p) for p in drop_patterns) or col in specific_drops)
]
data.drop(columns=cols_to_drop, inplace=True, errors='ignore')

cols_after = set(data.columns)
print(f"Đã loại bỏ {len(cols_before - cols_after)} cột. Còn lại: {data.shape[1]} cột")

# 3. XỬ LÝ GIÁ TRỊ THIẾU & CHUẨN HÓA
# Chuẩn hóa 'term' về số
if 'term' in data.columns:
    data['term'] = data['term'].astype(str).str.extract('(\d+)').astype(float)

# Điền 0 cho các cột đếm sự kiện (không có = 0)
fill_0 = ['delinq_2yrs', 'pub_rec', 'mort_acc', 'total_acc', 'recoveries', 'out_prncp', 'total_pymnt']
for col in fill_0:
    if col in data.columns:
        data[col] = data[col].fillna(0)

# Chuẩn hóa 'emp_length' về số từ 0 đến 10
def clean_emp_length(x):
    if pd.isna(x) or '< 1' in str(x): return 0
    if '10+' in str(x): return 10
    res = re.search('(\d+)', str(x))
    return int(res.group()) if res else 0

data['emp_length'] = data['emp_length'].apply(clean_emp_length)

# Điền median cho các cột tài chính liên tục
fill_median_cols = ['dti', 'revol_util', 'il_util', 'all_util', 'max_bal_bc', 'total_bc_limit', 'revol_bal']
for col in fill_median_cols:
    if col in data.columns:
        data[col] = data[col].fillna(data[col].median())

# Loại bỏ outlier cực đại của annual_inc (> percentile 99.9%)
# Giữ lại thu nhập hợp lý, tránh làm lệch phân tích trung bình
if 'annual_inc' in data.columns:
    threshold = data['annual_inc'].quantile(0.999)
    before = len(data)
    data = data[data['annual_inc'] < threshold]
    print(f"Loại bỏ outlier annual_inc (> {threshold:,.0f}): -{before - len(data):,} dòng")

# Loại bỏ dòng thiếu các cột quan trọng
critical_cols = ['annual_inc', 'loan_amnt', 'loan_status', 'issue_d', 'grade', 'int_rate']
before = len(data)
data.dropna(subset=[c for c in critical_cols if c in data.columns], inplace=True)
data.dropna(subset=['earliest_cr_line'], inplace=True)
print(f"Loại bỏ dòng null critical: -{before - len(data):,} dòng")

# 3b. CHUẨN HÓA NGÀY THÁNG
date_cols = ['issue_d', 'earliest_cr_line', 'last_pymnt_d']
for col in date_cols:
    data[col] = pd.to_datetime(data[col], format='%b-%Y', errors='coerce')

# Logic-based: điền last_pymnt_d thiếu bằng issue_d
data['last_pymnt_d'] = data['last_pymnt_d'].fillna(data['issue_d'])

# 4. TÁCH THỜI GIAN (FEATURE ENGINEERING)
data['issue_year'] = data['issue_d'].dt.year
data['issue_month'] = data['issue_d'].dt.month
data['issue_quarter'] = data['issue_d'].dt.quarter

# Chuyển ngày về dạng string YYYY-MM-DD trước khi xuất (tương thích SQL Server DATE)
for col in date_cols:
    data[col] = data[col].dt.strftime('%Y-%m-%d')

# 5. XUẤT FILE
# Chỉ giữ 33 cột theo từ điển dữ liệu
final_33 = [
    'id', 'loan_amnt', 'term', 'int_rate', 'grade', 'emp_length', 'home_ownership',
    'annual_inc', 'issue_d', 'loan_status', 'purpose', 'addr_state', 'dti',
    'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high',
    'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'out_prncp', 'total_pymnt',
    'recoveries', 'last_pymnt_d', 'mort_acc', 'total_bc_limit',
    'il_util', 'max_bal_bc', 'all_util',
    'issue_year', 'issue_month', 'issue_quarter'
]
existing_cols = [col for col in final_33 if col in data.columns]
missing_cols = [col for col in final_33 if col not in data.columns]
if missing_cols:
    print(f"⚠ Cột không tìm thấy: {missing_cols}")

data = data[existing_cols]

print(f"\n✓ Kích thước cuối cùng: {data.shape[0]:,} dòng × {data.shape[1]} cột")
data.to_csv('data_hqtcsdl.csv', index=False)
print("✓ Đã xuất file: data_hqtcsdl.csv")

Đã loại bỏ 121 cột. Còn lại: 30 cột
Loại bỏ outlier annual_inc (> 600,000): -2,592 dòng
Loại bỏ dòng null critical: -25 dòng

✓ Kích thước cuối cùng: 2,258,084 dòng × 33 cột
✓ Đã xuất file: data_hqtcsdl.csv


In [4]:
data.columns

Index(['id', 'loan_amnt', 'term', 'int_rate', 'grade', 'emp_length',
       'home_ownership', 'annual_inc', 'issue_d', 'loan_status', 'purpose',
       'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line',
       'fico_range_low', 'fico_range_high', 'pub_rec', 'revol_bal',
       'revol_util', 'total_acc', 'out_prncp', 'total_pymnt', 'recoveries',
       'last_pymnt_d', 'mort_acc', 'total_bc_limit', 'il_util', 'max_bal_bc',
       'all_util', 'issue_year', 'issue_month', 'issue_quarter'],
      dtype='object')

In [5]:
print(data.head(6))

         id  loan_amnt  term  int_rate grade  emp_length home_ownership  \
0  68407277     3600.0  36.0     13.99     C          10       MORTGAGE   
1  68355089    24700.0  36.0     11.99     C          10       MORTGAGE   
2  68341763    20000.0  60.0     10.78     B          10       MORTGAGE   
3  66310712    35000.0  60.0     14.85     C          10       MORTGAGE   
4  68476807    10400.0  60.0     22.45     F           3       MORTGAGE   
5  68426831    11950.0  36.0     13.44     C           4           RENT   

   annual_inc     issue_d loan_status  ... recoveries last_pymnt_d  mort_acc  \
0     55000.0  2015-12-01  Fully Paid  ...        0.0   2019-01-01       1.0   
1     65000.0  2015-12-01  Fully Paid  ...        0.0   2016-06-01       4.0   
2     63000.0  2015-12-01  Fully Paid  ...        0.0   2017-06-01       5.0   
3    110000.0  2015-12-01     Current  ...        0.0   2019-02-01       1.0   
4    104433.0  2015-12-01  Fully Paid  ...        0.0   2016-07-01       6

In [6]:
data.isnull().any()

id                  False
loan_amnt           False
term                False
int_rate            False
grade               False
emp_length          False
home_ownership      False
annual_inc          False
issue_d             False
loan_status         False
purpose             False
addr_state          False
dti                 False
delinq_2yrs         False
earliest_cr_line    False
fico_range_low      False
fico_range_high     False
pub_rec             False
revol_bal           False
revol_util          False
total_acc           False
out_prncp           False
total_pymnt         False
recoveries          False
last_pymnt_d        False
mort_acc            False
total_bc_limit      False
il_util             False
max_bal_bc          False
all_util            False
issue_year          False
issue_month         False
issue_quarter       False
dtype: bool